# ALIGN-PARETO — M1: Creative + Safety Prompt Classifier
Builds the four-quadrant prompt dataset from **Dolly-15k** and **WildGuardMix**, removes duplicates and benchmark
overlap, makes a family-safe 80/10/10 split, and trains two independent Logistic Regression heads on
`all-MiniLM-L6-v2` embeddings (spec §5–17, §36).

**Kaggle setup:** inputs `align-pareto-src` (code) and `align-pareto-raw` (published Dolly/WildGuardMix files) attached. CPU is enough.
All logic lives in `src/`; this notebook only runs it and shows the results.

In [ ]:
# 1. Environment
import os, sys, json, platform, shutil, glob
from pathlib import Path
import importlib.metadata as md

print("Python", platform.python_version())
try:
    import torch
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    else:
        print("No GPU (fine for M1)")
except ImportError:
    print("torch not installed")

In [ ]:
# 2. Dependencies (only what Kaggle lacks)
import subprocess
for pkg in ["datasketch", "sentence-transformers", "huggingface_hub", "seaborn"]:
    try:
        md.version(pkg)
    except md.PackageNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
for pkg in ["numpy", "pandas", "scipy", "torch", "sentence-transformers", "datasketch", "huggingface_hub", "matplotlib", "seaborn"]:
    try:
        print(f"{pkg:22s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:22s} MISSING")

In [ ]:
# 3. Project code: Kaggle dataset `align-pareto-src` -> /kaggle/working/align-pareto; locally, the repo itself
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/src/run_m1.py", recursive=True)
    assert hits, "Add the dataset 'align-pareto-src' to this notebook (Add Input)."
    ROOT = Path("/kaggle/working/align-pareto")
    shutil.copytree(Path(hits[0]).parents[1], ROOT, dirs_exist_ok=True)
else:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# 4. Raw data: attached Kaggle dataset `align-pareto-raw`, else Hugging Face download (WildGuardMix is gated)
from src.data.download import _local_copy, DOLLY_FILE
if _local_copy(DOLLY_FILE):
    print("Using attached raw-data copy:", _local_copy(DOLLY_FILE).parent)
elif ON_KAGGLE and not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("No HF_TOKEN secret available:", type(e).__name__)

In [ ]:
# 5. Run M1 (set QUICK=True for a fast smoke test)
QUICK = False
from src.run_m1 import run_m1
metrics = run_m1(ROOT, quick=QUICK)
RES = ROOT / "results" / ("m1_quick" if QUICK else "m1")
if ON_KAGGLE:
    shutil.rmtree(ROOT / "data/raw/hf_cache", ignore_errors=True)  # keep kernel output small

In [ ]:
# 6. Dataset audit
stats = json.load(open(RES / "dataset_statistics.json"))
for key in ["cleaning_stats", "exact_duplicates_removed", "unassigned_label_stats", "sampling_report",
            "wildguardtest_near_duplicates_removed", "near_duplicates_removed", "family_stats", "independence_check", "split_stats", "source_by_quadrant", "source_by_split"]:
    print(f"== {key}\n{json.dumps(stats[key], indent=2)}")

In [ ]:
# 7. Metrics
import pandas as pd
rows = {k: metrics[k] for k in ["creative_head", "safety_head", "safety_head_wildguardtest"]}
cols = ["threshold", "precision", "recall", "f1", "roc_auc", "pr_auc", "brier_score", "ece"]
display(pd.DataFrame(rows).T[cols])
print("Aggregate:", metrics["aggregate"])
print("Per-quadrant test accuracy:", json.dumps(metrics["per_quadrant_test_accuracy"], indent=2))
print("Error counts:", metrics["error_counts"])
print("Safety threshold (chosen vs max-F1):", json.dumps(metrics["safety_threshold_comparison"], indent=2))
print("By source (test):", json.dumps(metrics["by_source_test"], indent=2))
print("By adversarial:", json.dumps(metrics["by_adversarial"], indent=2))
print("Fictional-framing slice:", json.dumps(metrics["safety_head_fictional_framing_slice"], indent=2))

In [ ]:
# 8. Plots
from IPython.display import Image, display
for p in sorted((RES / "plots").glob("*.png")):
    print(p.name); display(Image(filename=str(p), width=520))

In [ ]:
# 9. Error analysis (spec §17): safety false negatives first
pd.set_option("display.max_colwidth", 200)
for name in ["safety_false_negatives", "safety_false_positives", "creative_false_negatives", "creative_false_positives"]:
    df = pd.read_csv(RES / f"errors_{name}.csv")
    print(f"== {name}: {len(df)}")
    display(df.head(10)[["source_dataset", "label_rule", "creative_probability", "safety_probability", "prompt"]])

## Next: creative-label audit
`results/m1/creative_rule_audit.csv` holds up to 100 random prompts per WildGuardMix rule outcome and per Dolly
creative category. `results/m1/fictional_framing_predictions.csv` holds the fictional-framing prompts that were
kept out of training. Fill in `human_creative_label` (1 / 0 / uncertain) by hand in both, then report the rule's
precision and the creative head's accuracy on the fictional-framing slice alongside these metrics.